In [ ]:
import pandas as pd
import os
import numpy as np
input_file = r"usa_00003.csv/usa_00003.csv"
output_file = "data_usa.csv"




# ==============================
# FILE PATHS

# ==============================
# REMOVE OLD FILE
# ==============================
if os.path.exists(output_file):
    os.remove(output_file)

# ==============================
# READ DATA IN CHUNKS (SAFE)
# ==============================
chunks = pd.read_csv(
    input_file,
    chunksize=200000,
    on_bad_lines='skip',   # handle malformed lines
    engine='python'
)

first_chunk = True
counter = 1

# ==============================
# MAPPING FUNCTIONS
# ==============================

def map_education(code):
    if code < 60:
        return "LowEducation"
    elif code < 70:
        return "Bachelor"
    else:
        return "Postgraduate"

def map_emp(x):
    if x == 1:
        return "Employed"
    elif x == 2:
        return "Unemployed"
    else:
        return "NotWorking"

# ==============================
# PROCESS EACH CHUNK
# ==============================
for chunk in chunks:

    # ------------------------------
    # CLEAN COLUMN NAMES
    # ------------------------------
    chunk.columns = chunk.columns.str.strip().str.upper()

    # ------------------------------
    # DATA VALIDATION (CRITICAL )
    # ------------------------------
    chunk["AGE"] = pd.to_numeric(chunk["AGE"], errors='coerce')
    chunk["INCWAGE"] = pd.to_numeric(chunk["INCWAGE"], errors='coerce')
    chunk["EMPSTAT"] = pd.to_numeric(chunk["EMPSTAT"], errors='coerce')
    chunk["EDUCD"] = pd.to_numeric(chunk["EDUCD"], errors='coerce')

    # remove invalid rows
    chunk = chunk.dropna(subset=["AGE", "INCWAGE", "EMPSTAT", "EDUCD"])

    # ------------------------------
    # FILTERING
    # ------------------------------
    chunk = chunk[(chunk["AGE"] >= 18) & (chunk["AGE"] < 100)]
    chunk = chunk[chunk["INCWAGE"] != 999999]
    chunk = chunk[chunk["INCWAGE"] > 0]

    # ------------------------------
    # TRANSFORMATIONS
    # ------------------------------
    chunk["employment_status"] = chunk["EMPSTAT"].apply(map_emp)
    chunk["education_level"] = chunk["EDUCD"].apply(map_education)

    # ------------------------------
    # RENAME COLUMNS
    # ------------------------------
    chunk = chunk.rename(columns={
        "AGE": "age",
        "INCWAGE": "income"
    })

    # ------------------------------
    # SELECT FINAL COLUMNS
    # ------------------------------
    chunk = chunk[["age", "income", "employment_status", "education_level"]]

    # ------------------------------
    # GENERATE UNIQUE ID
    # ------------------------------
    chunk["person_id"] = [
        "P" + str(i) for i in range(counter, counter + len(chunk))
    ]
    counter += len(chunk)

    chunk = chunk[
        ["person_id", "age", "income", "employment_status", "education_level"]
    ]

    # ------------------------------
    # SAVE CHUNK
    # ------------------------------
    chunk.to_csv(output_file, mode='a', index=False, header=first_chunk)
    first_chunk = False

print("✅ CLEANING + VALIDATION COMPLETED SUCCESSFULLY")

✅ CLEANING + VALIDATION COMPLETED SUCCESSFULLY


In [2]:
import pandas as pd
import numpy as np

input_file = r"data_usa.csv"
output_file = r"final_data.csv"

df = pd.read_csv(input_file)

expanded = []

for i in range(3):  # زود الرقم لو عايز حجم أكبر
    temp = df.copy()

    # random noise
    temp["age"] = temp["age"] + np.random.randint(-2, 3, len(temp))
    temp["income"] = temp["income"] * np.random.uniform(0.9, 1.1, len(temp))

    # constraints
    temp["age"] = temp["age"].clip(18, 70)
    temp["income"] = temp["income"].astype(int)

    expanded.append(temp)

df_big = pd.concat(expanded, ignore_index=True)

# إعادة ID
df_big["person_id"] = ["P" + str(i+1) for i in range(len(df_big))]

df_big.to_csv(output_file, index=False)

print("✅ Big data generated")

✅ Big data generated


In [2]:
import pandas as pd

df = pd.read_csv(r"final_data.csv")

print(df.isnull().sum())
print(df.tail())

person_id            0
age                  0
income               0
employment_status    0
education_level      0
dtype: int64
          person_id  age  income employment_status education_level
60246835  P60246836   60  114438          Employed    Postgraduate
60246836  P60246837   55   97732          Employed    Postgraduate
60246837  P60246838   33    8624          Employed    LowEducation
60246838  P60246839   65   17208        NotWorking    Postgraduate
60246839  P60246840   64   32564          Employed    Postgraduate
